In [1]:
# GPU check
!nvidia-smi

Wed Nov 25 07:04:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Mount Drive and create project folder
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/ml_internship_project'
for sub in ['data', 'models', 'outputs', 'app']:
    os.makedirs(f'{PROJECT_DIR}/{sub}', exist_ok=True)
os.chdir(PROJECT_DIR)
print(os.getcwd())

Mounted at /content/drive
/content/drive/MyDrive/ml_internship_project


In [3]:
# Install missing dependency
!pip install -q optuna

# Imports
import os
import warnings
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             mean_squared_error, mean_absolute_error, r2_score)
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
import shap
import optuna

import torch
import tensorflow as tf

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
tf.random.set_seed(SEED)
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

In [4]:
# Reinstall libraries and verify versions
!pip install -q streamlit pyngrok plotly joblib
import importlib.metadata as md, sklearn, sys
print('Python:', sys.version.split()[0])
for p in ['streamlit', 'pandas', 'numpy', 'scikit-learn', 'joblib', 'plotly']:
    print(p, md.version(p))

Python: 3.13.16
streamlit 1.67.0
pandas 2.2.3
numpy 2.4.6
scikit-learn 1.6.1
joblib 1.6.0
plotly 5.24.1


In [5]:
# Confirm GPU in PyTorch and TensorFlow
print('PyTorch CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')
print('TensorFlow GPUs:', tf.config.list_physical_devices('GPU'))

PyTorch CUDA: True | Tesla T4
TensorFlow GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [6]:
# Save progress to Drive
def save_progress(obj, name, kind='df'):
    path = f'{PROJECT_DIR}/data/{name}.csv' if kind == 'df' else f'{PROJECT_DIR}/models/{name}.pkl'
    obj.to_csv(path, index=False) if kind == 'df' else joblib.dump(obj, path)
    print('Saved:', path)

In [7]:
# Save progress to Drive
def save_progress(obj, name, kind='df'):
    path = f'{PROJECT_DIR}/data/{name}.csv' if kind == 'df' else f'{PROJECT_DIR}/models/{name}.pkl'
    obj.to_csv(path, index=False) if kind == 'df' else joblib.dump(obj, path)
    print('Saved:', path)

In [8]:
# Install UCI dataset fetcher
!pip install -q ucimlrepo

In [9]:
# Load three datasets and save to Drive
from ucimlrepo import fetch_ucirepo
from sklearn.datasets import load_breast_cancer
from sklearn.svm import SVC

heart = fetch_ucirepo(id=45)
df_heart = heart.data.features.copy()
df_heart['target'] = (heart.data.targets['num'] > 0).astype(int)

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
cols = ['pregnancies', 'glucose', 'blood_pressure', 'skin_thickness', 'insulin', 'bmi', 'diabetes_pedigree', 'age', 'target']
df_diabetes = pd.read_csv(url, header=None, names=cols)

bc = load_breast_cancer(as_frame=True)
df_cancer = bc.frame.copy()
df_cancer['target'] = 1 - df_cancer['target']

datasets = {'heart': df_heart, 'diabetes': df_diabetes, 'cancer': df_cancer}
for name, d in datasets.items():
    d.to_csv(f'{PROJECT_DIR}/data/{name}_raw.csv', index=False)
    print(name, d.shape, '| positive rate:', round(d['target'].mean(), 3), '| missing:', int(d.isna().sum().sum()))

heart (303, 14) | positive rate: 0.459 | missing: 6
diabetes (768, 9) | positive rate: 0.349 | missing: 0
cancer (569, 31) | positive rate: 0.373 | missing: 0


In [10]:
# Convert invalid zeros to NaN and save cleaned datasets
zero_cols = ['glucose', 'blood_pressure', 'skin_thickness', 'insulin', 'bmi']
df_diabetes[zero_cols] = df_diabetes[zero_cols].replace(0, np.nan)

datasets = {'heart': df_heart, 'diabetes': df_diabetes, 'cancer': df_cancer}
for name, d in datasets.items():
    d.to_csv(f'{PROJECT_DIR}/data/{name}_clean.csv', index=False)
    miss = d.isna().sum()
    print(f'\n{name}', d.shape)
    print(miss[miss > 0].to_string() if miss.sum() else 'no missing values')


heart (303, 14)
ca      4
thal    2

diabetes (768, 9)
glucose             5
blood_pressure     35
skin_thickness    227
insulin           374
bmi                11

cancer (569, 31)
no missing values


In [11]:
# Profile target balance and feature types
for name, d in datasets.items():
    print(f'\n{name}')
    print(d['target'].value_counts().to_string())
    print('dtypes:', d.dtypes.value_counts().to_dict())
    print('unique values per column (heart only):' if name == 'heart' else '', d.nunique().to_dict() if name == 'heart' else '')


heart
target
0    164
1    139
dtypes: {dtype('int64'): 11, dtype('float64'): 3}
unique values per column (heart only): {'age': 41, 'sex': 2, 'cp': 4, 'trestbps': 50, 'chol': 152, 'fbs': 2, 'restecg': 3, 'thalach': 91, 'exang': 2, 'oldpeak': 40, 'slope': 3, 'ca': 4, 'thal': 3, 'target': 2}

diabetes
target
0    500
1    268
dtypes: {dtype('float64'): 6, dtype('int64'): 3}
 

cancer
target
0    357
1    212
dtypes: {dtype('float64'): 30, dtype('int64'): 1}
 


In [12]:
# Define preprocessors and the four models
heart_num = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
heart_cat = ['cp', 'restecg', 'slope', 'ca', 'thal']
heart_bin = ['sex', 'fbs', 'exang']

def build_preprocessor(name):
    if name == 'heart':
        return ColumnTransformer([
            ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), heart_num),
            ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), heart_cat),
            ('bin', 'passthrough', heart_bin)
        ])
    return Pipeline([('imp', SimpleImputer(strategy='median', add_indicator=True)), ('sc', StandardScaler())])

def get_models(pos_weight):
    return {
        'SVM': SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=SEED),
        'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED),
        'Random Forest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=SEED),
        'XGBoost': XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, scale_pos_weight=pos_weight, eval_metric='logloss', random_state=SEED)
    }

In [13]:
# Stratified split and 5-fold CV across all datasets and models
from sklearn.model_selection import cross_validate

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
splits, results = {}, []

for name, d in datasets.items():
    X, y = d.drop(columns='target'), d['target']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
    splits[name] = (X_tr, X_te, y_tr, y_te)
    pw = (y_tr == 0).sum() / (y_tr == 1).sum()
    for mname, m in get_models(pw).items():
        pipe = Pipeline([('prep', build_preprocessor(name)), ('model', m)])
        cv = cross_validate(pipe, X_tr, y_tr, cv=skf, scoring=scoring, n_jobs=-1)
        row = {'dataset': name, 'model': mname}
        row.update({s: cv[f'test_{s}'].mean() for s in scoring})
        row['recall_std'] = cv['test_recall'].std()
        results.append(row)

df_results = pd.DataFrame(results).round(3)
df_results.to_csv(f'{PROJECT_DIR}/outputs/cv_results.csv', index=False)
joblib.dump(splits, f'{PROJECT_DIR}/data/splits.pkl')
print(df_results.to_string(index=False))

 dataset               model  accuracy  precision  recall    f1  roc_auc  recall_std
   heart                 SVM     0.822      0.819   0.801 0.806    0.887       0.048
   heart Logistic Regression     0.855      0.868   0.819 0.838    0.903       0.065
   heart       Random Forest     0.830      0.838   0.792 0.809    0.895       0.085
   heart             XGBoost     0.785      0.759   0.783 0.770    0.855       0.046
diabetes                 SVM     0.744      0.613   0.734 0.667    0.833       0.030
diabetes Logistic Regression     0.775      0.662   0.734 0.695    0.839       0.037
diabetes       Random Forest     0.774      0.719   0.589 0.645    0.827       0.031
diabetes             XGBoost     0.736      0.612   0.673 0.640    0.806       0.041
  cancer                 SVM     0.969      0.965   0.953 0.958    0.995       0.035
  cancer Logistic Regression     0.971      0.965   0.959 0.962    0.995       0.030
  cancer       Random Forest     0.963      0.959   0.941 0.949  

In [14]:
# Hyperparameter tuning with GridSearchCV
from sklearn.model_selection import GridSearchCV

grids = {
    'SVM': {'model__C': [0.1, 1, 10], 'model__gamma': ['scale', 0.01, 0.1]},
    'Logistic Regression': {'model__C': [0.01, 0.1, 1, 10]},
    'Random Forest': {'model__max_depth': [3, 5, None], 'model__min_samples_leaf': [1, 3, 5]},
    'XGBoost': {'model__max_depth': [2, 3, 4], 'model__learning_rate': [0.03, 0.1], 'model__n_estimators': [100, 300]}
}

tuned, rows = {}, []
for name in datasets:
    X_tr, X_te, y_tr, y_te = splits[name]
    pw = (y_tr == 0).sum() / (y_tr == 1).sum()
    for mname, m in get_models(pw).items():
        pipe = Pipeline([('prep', build_preprocessor(name)), ('model', m)])
        gs = GridSearchCV(pipe, grids[mname], cv=skf, scoring='roc_auc', n_jobs=-1)
        gs.fit(X_tr, y_tr)
        tuned[(name, mname)] = gs.best_estimator_
        base = df_results[(df_results.dataset == name) & (df_results.model == mname)]['roc_auc'].iloc[0]
        rows.append({'dataset': name, 'model': mname, 'untuned_auc': base, 'tuned_auc': round(gs.best_score_, 3),
                     'best_params': {k.replace('model__', ''): v for k, v in gs.best_params_.items()}})

df_tuned = pd.DataFrame(rows)
df_tuned.to_csv(f'{PROJECT_DIR}/outputs/tuned_results.csv', index=False)
joblib.dump(tuned, f'{PROJECT_DIR}/models/tuned_models.pkl')
print(df_tuned.to_string(index=False))

 dataset               model  untuned_auc  tuned_auc                                                  best_params
   heart                 SVM        0.887      0.902                                     {'C': 10, 'gamma': 0.01}
   heart Logistic Regression        0.903      0.903                                                     {'C': 1}
   heart       Random Forest        0.895      0.902                      {'max_depth': 5, 'min_samples_leaf': 3}
   heart             XGBoost        0.855      0.887 {'learning_rate': 0.03, 'max_depth': 2, 'n_estimators': 100}
diabetes                 SVM        0.833      0.839                                     {'C': 10, 'gamma': 0.01}
diabetes Logistic Regression        0.839      0.841                                                   {'C': 0.1}
diabetes       Random Forest        0.827      0.837                      {'max_depth': 5, 'min_samples_leaf': 5}
diabetes             XGBoost        0.806      0.833 {'learning_rate': 0.03, 'max_depth'

In [15]:
# Threshold tuning on out-of-fold predictions
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve

targets = {'heart': 0.90, 'diabetes': 0.85, 'cancer': 0.97}
thresholds, rows = {}, []

for name in datasets:
    X_tr, X_te, y_tr, y_te = splits[name]
    model = tuned[(name, 'Logistic Regression')]
    proba = cross_val_predict(model, X_tr, y_tr, cv=skf, method='predict_proba', n_jobs=-1)[:, 1]
    prec, rec, thr = precision_recall_curve(y_tr, proba)
    t = float(thr[np.where(rec[:-1] >= targets[name])[0][-1]])
    thresholds[name] = t
    row = {'dataset': name, 'threshold': round(t, 3)}
    for label, cut in [('at_0.5', 0.5), ('at_chosen', t)]:
        pred = (proba >= cut).astype(int)
        row[f'recall_{label}'] = round(recall_score(y_tr, pred), 3)
        row[f'precision_{label}'] = round(precision_score(y_tr, pred), 3)
        row[f'healthy_flagged_%_{label}'] = round(100 * ((pred == 1) & (y_tr == 0)).sum() / (y_tr == 0).sum(), 1)
    rows.append(row)

df_thr = pd.DataFrame(rows)
df_thr.to_csv(f'{PROJECT_DIR}/outputs/threshold_results.csv', index=False)
joblib.dump(thresholds, f'{PROJECT_DIR}/models/thresholds.pkl')
print(df_thr.T.to_string(header=False))

dataset                      heart  diabetes  cancer
threshold                    0.288     0.359   0.436
recall_at_0.5                 0.82     0.715   0.959
precision_at_0.5             0.858      0.64   0.964
healthy_flagged_%_at_0.5      11.5      21.5     2.1
recall_at_chosen             0.901      0.85   0.971
precision_at_chosen           0.69      0.54   0.959
healthy_flagged_%_at_chosen   34.4      38.8     2.5


In [16]:
# Final one-time test-set evaluation of Logistic Regression
rows = []
for name in datasets:
    X_tr, X_te, y_tr, y_te = splits[name]
    model = tuned[(name, 'Logistic Regression')]
    proba = model.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, proba)
    for label, cut in [('0.5', 0.5), ('chosen', thresholds[name])]:
        pred = (proba >= cut).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
        rows.append({'dataset': name, 'cutoff': label, 'threshold': round(cut, 3), 'roc_auc': round(auc, 3),
                     'recall': round(recall_score(y_te, pred), 3), 'precision': round(precision_score(y_te, pred), 3),
                     'f1': round(f1_score(y_te, pred), 3), 'caught_TP': tp, 'missed_FN': fn,
                     'false_alarms_FP': fp, 'correct_healthy_TN': tn})

df_test = pd.DataFrame(rows)
df_test.to_csv(f'{PROJECT_DIR}/outputs/final_test_results.csv', index=False)
print(df_test.to_string(index=False))

 dataset cutoff  threshold  roc_auc  recall  precision    f1  caught_TP  missed_FN  false_alarms_FP  correct_healthy_TN
   heart    0.5      0.500    0.965   0.929      0.812 0.867         26          2                6                  27
   heart chosen      0.288    0.965   0.964      0.730 0.831         27          1               10                  23
diabetes    0.5      0.500    0.816   0.685      0.597 0.638         37         17               25                  75
diabetes chosen      0.359    0.816   0.889      0.565 0.691         48          6               37                  63
  cancer    0.5      0.500    0.995   0.952      0.976 0.964         40          2                1                  71
  cancer chosen      0.436    0.995   0.952      0.976 0.964         40          2                1                  71


In [17]:
# Test-set comparison of all four tuned models at default cutoff
rows = []
for (name, mname), model in tuned.items():
    X_tr, X_te, y_tr, y_te = splits[name]
    proba = model.predict_proba(X_te)[:, 1]
    pred = (proba >= 0.5).astype(int)
    rows.append({'dataset': name, 'model': mname, 'accuracy': round(accuracy_score(y_te, pred), 3),
                 'recall': round(recall_score(y_te, pred), 3), 'precision': round(precision_score(y_te, pred), 3),
                 'roc_auc': round(roc_auc_score(y_te, proba), 3)})

df_test_all = pd.DataFrame(rows)
df_test_all.to_csv(f'{PROJECT_DIR}/outputs/final_test_all_models.csv', index=False)
print(df_test_all.to_string(index=False))

 dataset               model  accuracy  recall  precision  roc_auc
   heart                 SVM     0.869   0.929      0.812    0.952
   heart Logistic Regression     0.869   0.929      0.812    0.965
   heart       Random Forest     0.885   0.929      0.839    0.947
   heart             XGBoost     0.885   0.893      0.862    0.948
diabetes                 SVM     0.708   0.537      0.592    0.819
diabetes Logistic Regression     0.727   0.685      0.597    0.816
diabetes       Random Forest     0.747   0.759      0.612    0.825
diabetes             XGBoost     0.773   0.833      0.634    0.816
  cancer                 SVM     0.982   0.952      1.000    0.997
  cancer Logistic Regression     0.974   0.952      0.976    0.995
  cancer       Random Forest     0.965   0.905      1.000    0.997
  cancer             XGBoost     0.974   0.929      1.000    0.991


In [18]:
# Save final model bundle for the app
import sklearn
bundle = {}
for name in datasets:
    X_tr, X_te, y_tr, y_te = splits[name]
    full = pd.concat([X_tr, X_te])
    bundle[name] = {
        'model': tuned[(name, 'Logistic Regression')],
        'threshold': thresholds[name],
        'features': list(X_tr.columns),
        'stats': {c: {'min': float(full[c].min()), 'max': float(full[c].max()), 'median': float(full[c].median())} for c in X_tr.columns}
    }
joblib.dump(bundle, f'{PROJECT_DIR}/models/app_bundle.pkl')
print('sklearn version:', sklearn.__version__)
print({k: (len(v['features']), round(v['threshold'], 3)) for k, v in bundle.items()})

sklearn version: 1.6.1
{'heart': (13, 0.288), 'diabetes': (8, 0.359), 'cancer': (30, 0.436)}


In [19]:
# Top Logistic Regression coefficients per dataset
for name in datasets:
    pipe = tuned[(name, 'Logistic Regression')]
    names = pipe.named_steps['prep'].get_feature_names_out()
    coef = pipe.named_steps['model'].coef_[0]
    s = pd.Series(coef, index=names).sort_values(key=abs, ascending=False).head(6)
    print(f'\n{name}')
    print(s.round(2).to_string())


heart
cat__ca_0.0     -1.48
bin__sex         1.27
cat__cp_4.0      1.03
cat__thal_7.0    0.83
cat__ca_2.0      0.62
cat__cp_3.0     -0.59

diabetes
glucose                            1.03
bmi                                0.61
pregnancies                        0.31
diabetes_pedigree                  0.28
age                                0.19
missingindicator_blood_pressure    0.17

cancer
worst texture          1.47
radius error           1.31
worst symmetry         1.08
mean concave points    0.97
compactness error     -0.96
worst concavity        0.94


In [20]:
 # Create Streamlit theme config
os.makedirs(f'{PROJECT_DIR}/app/.streamlit', exist_ok=True)
with open(f'{PROJECT_DIR}/app/.streamlit/config.toml', 'w') as f:
    f.write('[theme]\nbase = "light"\nprimaryColor = "#1F5C6B"\nbackgroundColor = "#FAF9F6"\nsecondaryBackgroundColor = "#EFEDE7"\ntextColor = "#1C1C1C"\nfont = "sans serif"\n')

In [21]:
%%writefile /content/drive/MyDrive/ml_internship_project/app/app.py
# Streamlit app: disease risk screening
import numpy as np
import pandas as pd
import joblib
import streamlit as st
import plotly.graph_objects as go

PROJECT_DIR = '/content/drive/MyDrive/ml_internship_project'
ACCENT, RISK, GREY, INK = '#1F5C6B', '#B5472F', '#D9D6CE', '#1C1C1C'

st.set_page_config(page_title='Disease Risk Screening', layout='wide')
st.markdown("""<style>
.block-container {padding-top: 2rem; max-width: 1150px;}
h1, h2, h3 {font-weight: 600; letter-spacing: -0.01em;}
[data-testid="stMetricValue"] {font-size: 2.8rem;}
</style>""", unsafe_allow_html=True)

NAMES = {'heart': 'Heart disease', 'diabetes': 'Diabetes', 'cancer': 'Breast cancer'}
POP = {
    'heart': 'Trained on 303 patients from the Cleveland clinic (UCI), mostly older adults referred for suspected heart problems.',
    'diabetes': 'Trained on 768 women of Pima Indian heritage aged 21+. Not valid for men or other populations.',
    'cancer': 'Trained on 569 fine-needle-aspirate samples (Wisconsin). Inputs are lab measurements of cell nuclei.'
}
OPTIONAL = {'blood_pressure', 'skin_thickness', 'insulin', 'bmi'}
DEFAULT_NA = {'insulin', 'skin_thickness'}
SPECS = {
    'heart': {
        'age': ('int', 'Age (years)'),
        'sex': ('cat', 'Sex', {0: 'Female', 1: 'Male'}),
        'cp': ('cat', 'Chest pain type', {1: 'Typical angina', 2: 'Atypical angina', 3: 'Non-anginal pain', 4: 'Asymptomatic'}),
        'trestbps': ('int', 'Resting blood pressure (mm Hg)'),
        'chol': ('int', 'Serum cholesterol (mg/dl)'),
        'fbs': ('cat', 'Fasting blood sugar above 120 mg/dl', {0: 'No', 1: 'Yes'}),
        'restecg': ('cat', 'Resting ECG', {0: 'Normal', 1: 'ST-T abnormality', 2: 'LV hypertrophy'}),
        'thalach': ('int', 'Maximum heart rate achieved'),
        'exang': ('cat', 'Exercise-induced angina', {0: 'No', 1: 'Yes'}),
        'oldpeak': ('num', 'ST depression (oldpeak)'),
        'slope': ('cat', 'ST slope', {1: 'Upsloping', 2: 'Flat', 3: 'Downsloping'}),
        'ca': ('cat', 'Major vessels colored by fluoroscopy', {0: '0', 1: '1', 2: '2', 3: '3'}),
        'thal': ('cat', 'Thalassemia', {3: 'Normal', 6: 'Fixed defect', 7: 'Reversible defect'})
    },
    'diabetes': {
        'pregnancies': ('int', 'Pregnancies'),
        'glucose': ('int', 'Plasma glucose (mg/dl)'),
        'blood_pressure': ('int', 'Diastolic blood pressure (mm Hg)'),
        'skin_thickness': ('int', 'Skin fold thickness (mm)'),
        'insulin': ('int', 'Serum insulin (mu U/ml)'),
        'bmi': ('num', 'BMI'),
        'diabetes_pedigree': ('num', 'Diabetes pedigree function'),
        'age': ('int', 'Age (years)')
    }
}


@st.cache_resource
def load_all():
    bundle = joblib.load(f'{PROJECT_DIR}/models/app_bundle.pkl')
    splits = joblib.load(f'{PROJECT_DIR}/data/splits.pkl')
    return bundle, splits


@st.cache_data
def load_tables():
    return {k: pd.read_csv(f'{PROJECT_DIR}/outputs/{k}.csv') for k in ['cv_results', 'final_test_results', 'final_test_all_models']}


BUNDLE, SPLITS = load_all()
T = load_tables()


def spec(name, f):
    return SPECS.get(name, {}).get(f, ('num', f.title()))


def load_patient(name):
    X_te, y_te = SPLITS[name][1], SPLITS[name][3]
    i = np.random.randint(len(X_te))
    row = X_te.iloc[i]
    for f in BUNDLE[name]['features']:
        kind, key, v = spec(name, f)[0], f'{name}_{f}', row[f]
        s = BUNDLE[name]['stats'][f]
        if pd.isna(v):
            if f in OPTIONAL:
                st.session_state[f'{key}_na'] = True
            continue
        if f in OPTIONAL:
            st.session_state[f'{key}_na'] = False
        if kind == 'cat':
            st.session_state[key] = int(v)
        elif kind == 'int':
            st.session_state[key] = int(min(max(v, s['min']), s['max']))
        else:
            st.session_state[key] = float(min(max(v, s['min']), s['max']))
    st.session_state[f'{name}_truth'] = int(y_te.iloc[i])


def reset(name):
    for k in [k for k in st.session_state if k.startswith(f'{name}_')]:
        del st.session_state[k]


def render_feature(name, f, col):
    kind, label, *rest = spec(name, f)
    s = BUNDLE[name]['stats'][f]
    key = f'{name}_{f}'
    if kind == 'cat':
        opts = rest[0]
        keys = list(opts)
        kw = {} if key in st.session_state else {'index': keys.index(min(keys, key=lambda o: abs(o - s['median'])))}
        return float(col.selectbox(label, keys, format_func=opts.get, key=key, **kw))
    na_key = f'{key}_na'
    optional = f in OPTIONAL
    na = st.session_state.get(na_key, f in DEFAULT_NA) if optional else False
    if kind == 'int':
        lo, hi, med, step = int(s['min']), int(s['max']), int(round(s['median'])), 1
    else:
        lo, hi, med = s['min'], s['max'], s['median']
        step = (hi - lo) / 200
    kw = {} if key in st.session_state else {'value': med}
    v = col.slider(label, lo, hi, step=step, key=key, disabled=na, **kw)
    if optional:
        kw2 = {} if na_key in st.session_state else {'value': f in DEFAULT_NA}
        na = col.checkbox('Not measured', key=na_key, **kw2)
        return np.nan if na else float(v)
    return float(v)


def score(name, values):
    pipe = BUNDLE[name]['model']
    X = pd.DataFrame([values], columns=BUNDLE[name]['features']).astype(float)
    p = float(pipe.predict_proba(X)[0, 1])
    prep = pipe.named_steps['prep']
    Xt = prep.transform(X)
    Xt = Xt.toarray() if hasattr(Xt, 'toarray') else np.asarray(Xt)
    contrib = pd.Series(Xt[0] * pipe.named_steps['model'].coef_[0], index=[n.split('__', 1)[-1] for n in prep.get_feature_names_out()])
    return p, contrib


def risk_bar(p, t):
    fig = go.Figure()
    fig.add_shape(type='rect', x0=0, x1=1, y0=0, y1=1, fillcolor=GREY, line_width=0)
    fig.add_shape(type='rect', x0=0, x1=p, y0=0, y1=1, fillcolor=RISK if p >= t else ACCENT, line_width=0)
    fig.add_shape(type='line', x0=t, x1=t, y0=-0.25, y1=1.25, line=dict(color=INK, width=2))
    fig.add_annotation(x=t, y=1.5, text=f'threshold {t:.2f}', showarrow=False, font=dict(size=12, color=INK))
    fig.update_xaxes(range=[0, 1], tickformat='.0%', showgrid=False, zeroline=False)
    fig.update_yaxes(visible=False, range=[-0.4, 1.9])
    fig.update_layout(height=120, margin=dict(l=0, r=0, t=0, b=0), paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
    return fig


def contrib_chart(contrib):
    top = contrib.reindex(contrib.abs().sort_values(ascending=False).head(8).index).iloc[::-1]
    fig = go.Figure(go.Bar(x=top.values, y=top.index, orientation='h', marker_color=[RISK if v > 0 else ACCENT for v in top.values]))
    fig.update_layout(height=300, margin=dict(l=0, r=0, t=10, b=0), paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
                      xaxis_title='toward healthy  <-    ->  toward sick', showlegend=False)
    fig.update_xaxes(zeroline=True, zerolinecolor=INK, gridcolor=GREY)
    return fig


def tradeoff(name, t):
    y = SPLITS[name][3].values
    p = BUNDLE[name]['model'].predict_proba(SPLITS[name][1])[:, 1]
    grid = np.linspace(0.05, 0.95, 91)
    rec = [((p >= g) & (y == 1)).sum() / (y == 1).sum() for g in grid]
    fa = [((p >= g) & (y == 0)).sum() / (y == 0).sum() for g in grid]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=grid, y=rec, name='Sick patients caught', line=dict(color=ACCENT, width=3)))
    fig.add_trace(go.Scatter(x=grid, y=fa, name='Healthy patients falsely flagged', line=dict(color=RISK, width=3)))
    fig.add_vline(x=t, line_color=INK, line_width=2)
    fig.update_yaxes(tickformat='.0%', range=[0, 1.02], gridcolor=GREY)
    fig.update_xaxes(title='Flag threshold', gridcolor=GREY)
    fig.update_layout(height=340, margin=dict(l=0, r=0, t=10, b=0), paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
                      legend=dict(orientation='h', y=-0.3))
    caught, n_sick = int(((p >= t) & (y == 1)).sum()), int((y == 1).sum())
    flagged, n_healthy = int(((p >= t) & (y == 0)).sum()), int((y == 0).sum())
    return fig, caught, n_sick, flagged, n_healthy


with st.sidebar:
    st.title('Disease Risk Screening')
    name = st.radio('Condition', list(NAMES), format_func=NAMES.get)
    tuned_t = BUNDLE[name]['threshold']
    t = st.slider('Flag threshold', 0.05, 0.95, round(tuned_t, 2), 0.01, key=f'thr_{name}')
    st.caption(f'Tuned threshold: {tuned_t:.2f}. Lower catches more sick patients but raises false alarms.')
    st.divider()
    st.caption('Educational project. Not a diagnostic tool and not medical advice.')

st.title(NAMES[name])
st.caption(POP[name])
tab1, tab2, tab3 = st.tabs(['Predict', 'Model performance', 'About'])

with tab1:
    result = st.container()
    b1, b2, _ = st.columns([1.3, 1, 4])
    b1.button('Load a test patient', on_click=load_patient, args=(name,))
    b2.button('Reset', on_click=reset, args=(name,))
    feats = BUNDLE[name]['features']
    values = {}
    if name == 'cancer':
        groups = {
            'Mean measurements': [f for f in feats if f.startswith('mean')],
            'Error measurements': [f for f in feats if 'error' in f],
            'Worst measurements': [f for f in feats if f.startswith('worst')]
        }
        for gi, (g, fs) in enumerate(groups.items()):
            box = st.container() if gi == 0 else st.expander(g)
            with box:
                if gi == 0:
                    st.markdown(f'**{g}**')
                grid = st.columns(3)
                for i, f in enumerate(fs):
                    values[f] = render_feature(name, f, grid[i % 3])
    else:
        grid = st.columns(3)
        for i, f in enumerate(feats):
            values[f] = render_feature(name, f, grid[i % 3])

    p, contrib = score(name, values)
    flagged = p >= t
    with result:
        c1, c2 = st.columns(2)
        with c1:
            st.metric('Model score', f'{p:.0%}')
            colour = RISK if flagged else ACCENT
            text = 'Flagged: follow-up testing recommended' if flagged else 'Not flagged'
            st.markdown(f"<span style='color:{colour};font-weight:600;font-size:1.15rem'>{text}</span>", unsafe_allow_html=True)
            st.plotly_chart(risk_bar(p, t), width='stretch')
            st.caption('The score is not a calibrated probability. Flagged means worth a clinical follow-up test, not a diagnosis.')
            tk = f'{name}_truth'
            if tk in st.session_state:
                st.caption('Real outcome of the loaded test patient: ' + ('sick' if st.session_state[tk] else 'not sick') + ' (no longer applies once you edit inputs).')
        with c2:
            st.markdown('**What moved this score**')
            st.plotly_chart(contrib_chart(contrib), width='stretch')
            if name == 'cancer':
                st.caption('Cancer measurements are highly correlated, so individual contributions are unstable. Do not read them as causes.')

with tab2:
    st.subheader('Threshold trade-off on held-out test patients')
    fig, caught, n_sick, flagged_n, n_healthy = tradeoff(name, t)
    st.plotly_chart(fig, width='stretch')
    st.markdown(f'At threshold **{t:.2f}**: catches **{caught} of {n_sick}** sick patients and falsely flags **{flagged_n} of {n_healthy}** healthy patients.')
    st.caption('The test set is small, so every number here has a margin of error of several points.')
    st.subheader('Cross-validated comparison of the four algorithms (training data, 5-fold)')
    cv = T['cv_results']
    st.dataframe(cv[cv.dataset == name].drop(columns='dataset'), hide_index=True)
    st.caption('If the gap between two models is smaller than recall_std, the difference is noise.')
    st.subheader('Final test: all four tuned models at the default 0.5 cutoff')
    ta = T['final_test_all_models']
    st.dataframe(ta[ta.dataset == name].drop(columns='dataset'), hide_index=True)
    st.caption('Differences in recall here mostly reflect how each model sets its probabilities, not better separation of patients.')
    st.subheader('Final test: chosen model (Logistic Regression) at both cutoffs')
    tf = T['final_test_results']
    st.dataframe(tf[tf.dataset == name].drop(columns='dataset'), hide_index=True)

with tab3:
    st.markdown("""
**Data.** UCI Heart Disease (Cleveland, 303 patients), Pima Indians Diabetes (768 women), Breast Cancer Wisconsin Diagnostic (569 samples).

**Method.** Missing values are imputed inside a pipeline fitted on training data only. Four algorithms (SVM, Logistic Regression, Random Forest, XGBoost) were compared with 5-fold cross-validation and tuned with grid search. Their scores were statistically indistinguishable, so the simplest and most explainable model, Logistic Regression, was chosen before the test set was opened. The flag threshold was tuned on out-of-fold predictions to reach a recall target.

**Limitations.**
Small datasets, so test results carry a wide margin of error.
Each model reflects its source population only.
Model scores are not calibrated probabilities.
Cancer: 2 of 42 malignant test samples were missed.
Nothing here is clinically validated.
""")

Overwriting /content/drive/MyDrive/ml_internship_project/app/app.py


In [22]:
# Build self-contained GitHub-ready repo folder
import shutil, sys, importlib.metadata as md

REPO = f'{PROJECT_DIR}/repo'
shutil.rmtree(REPO, ignore_errors=True)
for sub in ['models', 'data', 'outputs', '.streamlit']:
    os.makedirs(f'{REPO}/{sub}')

src = open(f'{PROJECT_DIR}/app/app.py').read()
src = src.replace("PROJECT_DIR = '/content/drive/MyDrive/ml_internship_project'", "from pathlib import Path\nPROJECT_DIR = str(Path(__file__).parent)")
assert 'Path(__file__)' in src
open(f'{REPO}/app.py', 'w').write(src)

shutil.copy(f'{PROJECT_DIR}/models/app_bundle.pkl', f'{REPO}/models/')
shutil.copy(f'{PROJECT_DIR}/data/splits.pkl', f'{REPO}/data/')
shutil.copy(f'{PROJECT_DIR}/app/.streamlit/config.toml', f'{REPO}/.streamlit/')
for f in os.listdir(f'{PROJECT_DIR}/outputs'):
    shutil.copy(f'{PROJECT_DIR}/outputs/{f}', f'{REPO}/outputs/')

pkgs = ['streamlit', 'pandas', 'numpy', 'scikit-learn', 'joblib', 'plotly']
reqs = '\n'.join(f'{p}=={md.version(p)}' for p in pkgs)
open(f'{REPO}/requirements.txt', 'w').write(reqs + '\n')
open(f'{REPO}/.gitignore', 'w').write('__pycache__/\n.ipynb_checkpoints/\n')

print('Python:', sys.version.split()[0])
print(reqs)
for root, _, files in os.walk(REPO):
    for f in files:
        print(os.path.relpath(f'{root}/{f}', REPO))

Python: 3.13.16
streamlit==1.67.0
pandas==2.2.3
numpy==2.4.6
scikit-learn==1.6.1
joblib==1.6.0
plotly==5.24.1
app.py
requirements.txt
.gitignore
models/app_bundle.pkl
data/splits.pkl
outputs/cv_results.csv
outputs/tuned_results.csv
outputs/threshold_results.csv
outputs/final_test_results.csv
outputs/final_test_all_models.csv
.streamlit/config.toml


In [23]:
# Write human-voice README and zip the repo
readme = r'''# Disease Risk Screening

A small app that flags possible heart disease, diabetes and breast cancer from patient data. I built it during my internship at [CodeAlpha/Machine Learning].

**Try it:** YOUR-LINK (if it's asleep, click "wake up" and give it a minute)

## Why I built it
My task was to predict diseases from medical data using SVM, Logistic Regression, Random Forest and XGBoost. I wanted the result to be something you can actually click around in, not just a notebook with numbers.

## What you can do in the app
- Pick a condition and enter patient values, or load a real test patient.
- Drag the threshold and see sick patients caught vs healthy people falsely flagged.
- See which inputs pushed the score up or down.
- Compare all four algorithms.

## Problems I hit and what I did
- **Diabetes data had zeros that weren't real.** A glucose of 0 means "not measured". I turned them into missing values and handled them properly. Insulin was almost half missing.
- **Easy to cheat by accident.** Filling missing values before splitting leaks test data into training. I did all preprocessing inside a pipeline.
- **The four algorithms scored almost the same.** Differences were smaller than the noise, so I picked Logistic Regression because it's simple and explainable.
- **The default cutoff missed too many sick people.** On diabetes it missed 17 of 54 test patients. Lowering the cutoff cut that to 6, but 12 more healthy people got flagged. That trade-off is the slider in the app.
- **XGBoost looked best on one test table.** It wasn't. Its ranking ability (AUC) was identical, it just used a different cutoff. I didn't switch models.
- **My Colab demo link kept dying.** I moved the app to GitHub and Streamlit Cloud so the link stays up.

## Results (held-out test set)
| Dataset | Test patients | AUC | Sick caught |
|---|---|---|---|
| Heart | 61 | 0.965 | 27 of 28 |
| Diabetes | 154 | 0.816 | 48 of 54 |
| Cancer | 114 | 0.995 | 40 of 42 |

Heart's cross-validated AUC was about 0.90, so 0.965 is probably a lucky small test set. On cancer I aimed for higher recall and missed 2 tumors.

## Tech stack
- **Python** on Google Colab (GPU wasn't needed)
- **pandas, NumPy** for data
- **scikit-learn** for pipelines, SVM, Logistic Regression, Random Forest, cross-validation and tuning
- **XGBoost** for gradient boosting
- **Plotly** for charts
- **Streamlit** for the app, hosted on Streamlit Community Cloud
- **Data:** UCI Heart Disease, Pima Indians Diabetes, Breast Cancer Wisconsin

## Limits
Small datasets, so results are shaky. Each model only fits the group it was trained on (the diabetes model is Pima women only). Scores are not true probabilities. This is a learning project, not medical advice.

## Run it yourself
'''

import shutil
# Save the README.md to the repo folder
with open(f'{REPO}/README.md', 'w') as f:
    f.write(readme)

# Zip the repo folder
shutil.make_archive(f'{PROJECT_DIR}/disease_risk_screening_repo', 'zip', REPO)
print("Saved README.md to repo and created disease_risk_screening_repo.zip")

Saved README.md to repo and created disease_risk_screening_repo.zip


In [27]:
%%writefile /content/drive/MyDrive/ml_internship_project/repo/app.py
# Streamlit app: disease risk screening (v2 layout)
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import streamlit as st
import plotly.graph_objects as go

PROJECT_DIR = str(Path(__file__).parent)
ACCENT, RISK, GREY, INK = '#1F5C6B', '#B5472F', '#D9D6CE', '#1C1C1C'

st.set_page_config(page_title='Disease Risk Screening', layout='wide')
st.markdown("""<style>
.block-container {padding-top: 2rem; max-width: 1150px;}
h1, h2, h3 {font-weight: 600; letter-spacing: -0.01em;}
</style>""", unsafe_allow_html=True)

NAMES = {'heart': 'Heart disease', 'diabetes': 'Diabetes', 'cancer': 'Breast cancer'}
POP = {
    'heart': 'Trained on 303 patients from the Cleveland clinic (UCI), mostly older adults referred for suspected heart problems.',
    'diabetes': 'Trained on 768 women of Pima Indian heritage aged 21+. Not valid for men or other populations.',
    'cancer': 'Trained on 569 fine-needle-aspirate samples (Wisconsin). Inputs are lab measurements of cell nuclei.'
}
FLAG_TEXT = {'heart': 'Flagged: heart check-up recommended', 'diabetes': 'Flagged: blood sugar test recommended', 'cancer': 'Flagged: further tests recommended'}
OPTIONAL = {'blood_pressure', 'skin_thickness', 'insulin', 'bmi'}
DEFAULT_NA = {'insulin', 'skin_thickness'}
SPECS = {
    'heart': {
        'age': ('int', 'Age (years)'),
        'sex': ('cat', 'Sex', {0: 'Female', 1: 'Male'}),
        'cp': ('cat', 'Chest pain type', {1: 'Typical angina', 2: 'Atypical angina', 3: 'Non-anginal pain', 4: 'Asymptomatic'}),
        'trestbps': ('int', 'Resting blood pressure (mm Hg)'),
        'chol': ('int', 'Serum cholesterol (mg/dl)'),
        'fbs': ('cat', 'Fasting blood sugar above 120 mg/dl', {0: 'No', 1: 'Yes'}),
        'restecg': ('cat', 'Resting ECG', {0: 'Normal', 1: 'ST-T abnormality', 2: 'LV hypertrophy'}),
        'thalach': ('int', 'Maximum heart rate achieved'),
        'exang': ('cat', 'Exercise-induced angina', {0: 'No', 1: 'Yes'}),
        'oldpeak': ('num', 'ST depression (oldpeak)'),
        'slope': ('cat', 'ST slope', {1: 'Upsloping', 2: 'Flat', 3: 'Downsloping'}),
        'ca': ('cat', 'Major vessels colored by fluoroscopy', {0: '0', 1: '1', 2: '2', 3: '3'}),
        'thal': ('cat', 'Thalassemia', {3: 'Normal', 6: 'Fixed defect', 7: 'Reversible defect'})
    },
    'diabetes': {
        'pregnancies': ('int', 'Pregnancies'),
        'glucose': ('int', 'Plasma glucose (mg/dl)'),
        'blood_pressure': ('int', 'Diastolic blood pressure (mm Hg)'),
        'skin_thickness': ('int', 'Skin fold thickness (mm)'),
        'insulin': ('int', 'Serum insulin (mu U/ml)'),
        'bmi': ('num', 'BMI'),
        'diabetes_pedigree': ('num', 'Diabetes pedigree function'),
        'age': ('int', 'Age (years)')
    }
}


@st.cache_resource
def load_all():
    bundle = joblib.load(f'{PROJECT_DIR}/models/app_bundle.pkl')
    splits = joblib.load(f'{PROJECT_DIR}/data/splits.pkl')
    return bundle, splits


@st.cache_data
def load_tables():
    return {k: pd.read_csv(f'{PROJECT_DIR}/outputs/{k}.csv') for k in ['cv_results', 'final_test_results', 'final_test_all_models']}


BUNDLE, SPLITS = load_all()
T = load_tables()


def spec(name, f):
    return SPECS.get(name, {}).get(f, ('num', f.title()))


def pretty(name, n):
    if n.startswith('missingindicator_'):
        f = n[len('missingindicator_'):]
        return spec(name, f)[1].split(' (')[0] + ' (not measured)'
    base, _, val = n.rpartition('_')
    if name == 'heart' and base in SPECS['heart'] and SPECS['heart'][base][0] == 'cat':
        s = SPECS['heart'][base]
        try:
            return f'{s[1]}: {s[2].get(int(float(val)), val)}'
        except ValueError:
            pass
    if name in SPECS and n in SPECS[name]:
        return SPECS[name][n][1].split(' (')[0]
    return n.replace('_', ' ').capitalize()


def load_patient(name):
    X_te, y_te = SPLITS[name][1], SPLITS[name][3]
    i = np.random.randint(len(X_te))
    row = X_te.iloc[i]
    for f in BUNDLE[name]['features']:
        kind, key, v = spec(name, f)[0], f'{name}_{f}', row[f]
        s = BUNDLE[name]['stats'][f]
        if pd.isna(v):
            if f in OPTIONAL:
                st.session_state[f'{key}_na'] = True
            continue
        if f in OPTIONAL:
            st.session_state[f'{key}_na'] = False
        if kind == 'cat':
            st.session_state[key] = int(v)
        elif kind == 'int':
            st.session_state[key] = int(min(max(v, s['min']), s['max']))
        else:
            st.session_state[key] = float(min(max(v, s['min']), s['max']))
    st.session_state[f'{name}_truth'] = int(y_te.iloc[i])


def reset(name):
    for k in [k for k in st.session_state if k.startswith(f'{name}_')]:
        del st.session_state[k]


def render_feature(name, f, col):
    kind, label, *rest = spec(name, f)
    s = BUNDLE[name]['stats'][f]
    key = f'{name}_{f}'
    if kind == 'cat':
        opts = rest[0]
        keys = list(opts)
        kw = {} if key in st.session_state else {'index': keys.index(min(keys, key=lambda o: abs(o - s['median'])))}
        return float(col.selectbox(label, keys, format_func=opts.get, key=key, **kw))
    na_key = f'{key}_na'
    optional = f in OPTIONAL
    na = st.session_state.get(na_key, f in DEFAULT_NA) if optional else False
    if kind == 'int':
        lo, hi, med, step, fmt = int(s['min']), int(s['max']), int(round(s['median'])), 1, '%d'
    else:
        lo, hi, med = float(s['min']), float(s['max']), float(s['median'])
        e = int(np.floor(np.log10(max(hi - lo, 1e-9)))) - 2
        step, fmt = float(10 ** e), f'%.{max(0, -e)}f'
    kw = {} if key in st.session_state else {'value': med}
    v = col.number_input(label, min_value=lo, max_value=hi, step=step, format=fmt, key=key, disabled=na, **kw)
    if optional:
        kw2 = {} if na_key in st.session_state else {'value': f in DEFAULT_NA}
        na = col.checkbox('Not measured', key=na_key, **kw2)
        return np.nan if na else float(v)
    return float(v)


def score(name, values):
    pipe = BUNDLE[name]['model']
    X = pd.DataFrame([values], columns=BUNDLE[name]['features']).astype(float)
    p = float(pipe.predict_proba(X)[0, 1])
    prep = pipe.named_steps['prep']
    Xt = prep.transform(X)
    Xt = Xt.toarray() if hasattr(Xt, 'toarray') else np.asarray(Xt)
    raw = [n.split('__', 1)[-1] for n in prep.get_feature_names_out()]
    contrib = pd.Series(Xt[0] * pipe.named_steps['model'].coef_[0], index=[pretty(name, n) for n in raw])
    return p, contrib[contrib != 0]


def risk_bar(p, t):
    fig = go.Figure()
    fig.add_shape(type='rect', x0=0, x1=1, y0=0, y1=1, fillcolor=GREY, line_width=0)
    fig.add_shape(type='rect', x0=0, x1=p, y0=0, y1=1, fillcolor=RISK if p >= t else ACCENT, line_width=0)
    fig.add_shape(type='line', x0=t, x1=t, y0=-0.25, y1=1.25, line=dict(color=INK, width=2))
    fig.add_annotation(x=t, y=1.5, text=f'flag threshold {t:.2f}', showarrow=False, font=dict(size=12, color=INK))
    fig.update_xaxes(range=[0, 1], tickformat='.0%', showgrid=False, zeroline=False)
    fig.update_yaxes(visible=False, range=[-0.4, 1.9])
    fig.update_layout(height=120, margin=dict(l=0, r=0, t=0, b=0), paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
    return fig


def contrib_chart(contrib):
    top = contrib.reindex(contrib.abs().sort_values(ascending=False).head(6).index).iloc[::-1]
    fig = go.Figure(go.Bar(x=top.values, y=top.index, orientation='h', marker_color=[RISK if v > 0 else ACCENT for v in top.values]))
    fig.update_layout(height=280, margin=dict(l=0, r=0, t=10, b=0), paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
                      xaxis_title='lowers the score   <-    ->   raises the score', showlegend=False)
    fig.update_xaxes(zeroline=True, zerolinecolor=INK, gridcolor=GREY)
    return fig


def tradeoff(name, t):
    y = SPLITS[name][3].values
    p = BUNDLE[name]['model'].predict_proba(SPLITS[name][1])[:, 1]
    grid = np.linspace(0.05, 0.95, 91)
    rec = [((p >= g) & (y == 1)).sum() / (y == 1).sum() for g in grid]
    fa = [((p >= g) & (y == 0)).sum() / (y == 0).sum() for g in grid]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=grid, y=rec, name='Sick patients caught', line=dict(color=ACCENT, width=3)))
    fig.add_trace(go.Scatter(x=grid, y=fa, name='Healthy patients falsely flagged', line=dict(color=RISK, width=3)))
    fig.add_vline(x=t, line_color=INK, line_width=2)
    fig.update_yaxes(tickformat='.0%', range=[0, 1.02], gridcolor=GREY)
    fig.update_xaxes(title='Flag threshold', gridcolor=GREY)
    fig.update_layout(height=340, margin=dict(l=0, r=0, t=10, b=0), paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
                      legend=dict(orientation='h', y=-0.3))
    caught, n_sick = int(((p >= t) & (y == 1)).sum()), int((y == 1).sum())
    flagged, n_healthy = int(((p >= t) & (y == 0)).sum()), int((y == 0).sum())
    return fig, caught, n_sick, flagged, n_healthy


with st.sidebar:
    st.title('Disease Risk Screening')
    name = st.radio('Choose a condition', list(NAMES), format_func=NAMES.get)
    tuned_t = BUNDLE[name]['threshold']
    with st.expander('Advanced settings'):
        t = st.slider('Flag threshold', 0.05, 0.95, round(tuned_t, 2), 0.01, key=f'thr_{name}')
        st.caption(f'Tuned value: {tuned_t:.2f}. Lower catches more sick patients but raises false alarms.')
    st.divider()
    st.caption('Educational project. Not a diagnostic tool and not medical advice.')

if 't' not in dir():
    t = round(tuned_t, 2)

st.title(NAMES[name])
st.caption(POP[name])
tab1, tab2, tab3 = st.tabs(['Predict', 'Model performance', 'About'])

with tab1:
    left, right = st.columns([1.2, 1], gap='large')
    values = {}
    feats = BUNDLE[name]['features']
    with left:
        st.subheader('1. Patient details')
        b1, b2 = st.columns([2, 1])
        b1.button('Load a sample test patient', on_click=load_patient, args=(name,), use_container_width=True)
        b2.button('Reset', on_click=reset, args=(name,), use_container_width=True)
        st.caption('Start by loading a real held-out patient, or type your own values. Boxes accept only the range seen in the training data.')
        if name == 'cancer':
            groups = {
                'Mean measurements': [f for f in feats if f.startswith('mean')],
                'Error measurements': [f for f in feats if 'error' in f],
                'Worst measurements': [f for f in feats if f.startswith('worst')]
            }
            for gi, (g, fs) in enumerate(groups.items()):
                box = st.container() if gi == 0 else st.expander(g)
                with box:
                    if gi == 0:
                        st.markdown(f'**{g}**')
                    grid = st.columns(2)
                    for i, f in enumerate(fs):
                        values[f] = render_feature(name, f, grid[i % 2])
        else:
            grid = st.columns(2)
            for i, f in enumerate(feats):
                values[f] = render_feature(name, f, grid[i % 2])

    p, contrib = score(name, values)
    flagged = p >= t
    with right:
        st.subheader('2. Result')
        colour = RISK if flagged else ACCENT
        text = FLAG_TEXT[name] if flagged else 'Not flagged'
        st.markdown(
            f"<div style='border-left:6px solid {colour};background:#FFFFFF;padding:1rem 1.2rem;border-radius:6px'>"
            f"<div style='font-size:1.5rem;font-weight:600;color:{colour}'>{text}</div>"
            f"<div style='color:#555;margin-top:0.3rem'>Model score {p:.0%} (flag threshold {t:.0%})</div></div>",
            unsafe_allow_html=True)
        st.plotly_chart(risk_bar(p, t), width='stretch')
        st.caption('The score is not a true probability. Flagged means a follow-up test is worth doing, not that disease is confirmed.')
        tk = f'{name}_truth'
        if tk in st.session_state:
            st.info('Real outcome of the loaded test patient: ' + ('sick' if st.session_state[tk] else 'not sick') + '. This no longer applies once you edit inputs.')
        st.markdown('**What pushed the score**')
        if name != 'cancer':
            up = list(contrib[contrib > 0.05].sort_values(ascending=False).head(3).index)
            down = list(contrib[contrib < -0.05].sort_values().head(3).index)
            if up:
                st.markdown('Raised it: ' + ', '.join(up))
            if down:
                st.markdown('Lowered it: ' + ', '.join(down))
        st.plotly_chart(contrib_chart(contrib), width='stretch')
        if name == 'cancer':
            st.caption('Cancer measurements are highly correlated, so individual contributions are unstable. Do not read them as causes.')

with tab2:
    st.subheader('Threshold trade-off on held-out test patients')
    fig, caught, n_sick, flagged_n, n_healthy = tradeoff(name, t)
    st.plotly_chart(fig, width='stretch')
    st.markdown(f'At threshold **{t:.2f}**: catches **{caught} of {n_sick}** sick patients and falsely flags **{flagged_n} of {n_healthy}** healthy patients.')
    st.caption('The test set is small, so every number here has a margin of error of several points.')
    st.subheader('Cross-validated comparison of the four algorithms (training data, 5-fold)')
    cv = T['cv_results']
    st.dataframe(cv[cv.dataset == name].drop(columns='dataset'), hide_index=True)
    st.caption('If the gap between two models is smaller than recall_std, the difference is noise.')
    st.subheader('Final test: all four tuned models at the default 0.5 cutoff')
    ta = T['final_test_all_models']
    st.dataframe(ta[ta.dataset == name].drop(columns='dataset'), hide_index=True)
    st.caption('Differences in recall here mostly reflect how each model sets its probabilities, not better separation of patients.')
    st.subheader('Final test: chosen model (Logistic Regression) at both cutoffs')
    tf = T['final_test_results']
    st.dataframe(tf[tf.dataset == name].drop(columns='dataset'), hide_index=True)

with tab3:
    st.markdown("""
**Data.** UCI Heart Disease (Cleveland, 303 patients), Pima Indians Diabetes (768 women), Breast Cancer Wisconsin Diagnostic (569 samples).

**Method.** Missing values are imputed inside a pipeline fitted on training data only. Four algorithms (SVM, Logistic Regression, Random Forest, XGBoost) were compared with 5-fold cross-validation and tuned with grid search. Their scores were statistically indistinguishable, so the simplest and most explainable model, Logistic Regression, was chosen before the test set was opened. The flag threshold was tuned on out-of-fold predictions to reach a recall target.

**Limitations.**
Small datasets, so test results carry a wide margin of error.
Each model reflects its source population only.
Model scores are not calibrated probabilities.
Cancer: 2 of 42 malignant test samples were missed.
Nothing here is clinically validated.
""")

Overwriting /content/drive/MyDrive/ml_internship_project/repo/app.py


In [33]:
# Update theme colors for the new layout
with open(f'{PROJECT_DIR}/repo/.streamlit/config.toml', 'w') as f:
    f.write('[theme]\nbase = "light"\nprimaryColor = "#1E5B45"\nbackgroundColor = "#F6F7F4"\nsecondaryBackgroundColor = "#EEF0EA"\ntextColor = "#1C2420"\nfont = "sans serif"\n')

In [34]:
%%writefile /content/drive/MyDrive/ml_internship_project/repo/app.py
# Streamlit app: disease risk screening (v3, card layout)
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import streamlit as st
import plotly.graph_objects as go

PROJECT_DIR = str(Path(__file__).parent)
GREEN, RISK, GREY, INK = '#1E5B45', '#B5472F', '#DDE0D8', '#1C2420'

st.set_page_config(page_title='Disease Risk Screening', layout='wide', initial_sidebar_state='collapsed')
st.markdown("""<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');
.stApp, .stApp p, .stApp label, .stApp input {font-family: 'Inter', sans-serif;}
[data-testid="stSidebar"], [data-testid="stSidebarCollapsedControl"], [data-testid="stHeader"], footer {display: none;}
.block-container {padding-top: 1rem; max-width: 1100px;}
.st-key-card_left, .st-key-card_right {background: #FFFFFF; border-radius: 14px; padding: 0.6rem 0.8rem;}
.stButton button, [data-testid="stFormSubmitButton"] button {width: 100%; border-radius: 10px;}
[data-testid="stFormSubmitButton"] button {background: #1E5B45; color: #FFFFFF; border: 0; padding: 0.7rem 0;}
.nav {display: flex; justify-content: space-between; align-items: center; padding: 0.4rem 0 1rem; border-bottom: 1px solid #DDE0D8; margin-bottom: 1.2rem;}
.brand {font-weight: 700; font-size: 1.25rem; color: #1E5B45;}
.muted {color: #6B746E; font-size: 0.85rem;}
.verdict {font-size: 1.9rem; font-weight: 700; line-height: 1.2; margin: 0.2rem 0;}
.score {font-size: 1.1rem; color: #3D4640; margin-bottom: 0.6rem;}
.empty {text-align: center; padding: 3rem 1rem; color: #6B746E;}
</style>""", unsafe_allow_html=True)

NAMES = {'heart': 'Heart disease', 'diabetes': 'Diabetes', 'cancer': 'Breast cancer'}
DISEASE = {'heart': 'heart disease', 'diabetes': 'diabetes', 'cancer': 'breast cancer'}
POP = {
    'heart': 'Trained on 303 patients from the Cleveland clinic (UCI), mostly older adults referred for suspected heart problems.',
    'diabetes': 'Trained on 768 women of Pima Indian heritage aged 21+. Not valid for men or other populations.',
    'cancer': 'Trained on 569 fine-needle-aspirate samples (Wisconsin). Inputs are lab measurements of cell nuclei.'
}
SPECS = {
    'heart': {
        'age': ('int', 'Age (years)'),
        'sex': ('cat', 'Sex', {0: 'Female', 1: 'Male'}),
        'cp': ('cat', 'Chest pain type', {1: 'Typical angina', 2: 'Atypical angina', 3: 'Non-anginal pain', 4: 'Asymptomatic'}),
        'trestbps': ('int', 'Resting blood pressure (mm Hg)'),
        'chol': ('int', 'Cholesterol (mg/dl)'),
        'fbs': ('cat', 'Fasting blood sugar above 120 mg/dl', {0: 'No', 1: 'Yes'}),
        'restecg': ('cat', 'Resting ECG', {0: 'Normal', 1: 'ST-T abnormality', 2: 'LV hypertrophy'}),
        'thalach': ('int', 'Maximum heart rate (bpm)'),
        'exang': ('cat', 'Exercise-induced angina', {0: 'No', 1: 'Yes'}),
        'oldpeak': ('num', 'ST depression (oldpeak)'),
        'slope': ('cat', 'ST slope', {1: 'Upsloping', 2: 'Flat', 3: 'Downsloping'}),
        'ca': ('cat', 'Major vessels (fluoroscopy)', {0: '0', 1: '1', 2: '2', 3: '3'}),
        'thal': ('cat', 'Thalassemia', {3: 'Normal', 6: 'Fixed defect', 7: 'Reversible defect'})
    },
    'diabetes': {
        'pregnancies': ('int', 'Pregnancies'),
        'glucose': ('int', 'Plasma glucose (mg/dl)'),
        'blood_pressure': ('int', 'Diastolic blood pressure (mm Hg)'),
        'skin_thickness': ('int', 'Skin fold thickness (mm)'),
        'insulin': ('int', 'Serum insulin (mu U/ml)'),
        'bmi': ('num', 'BMI'),
        'diabetes_pedigree': ('num', 'Diabetes pedigree function'),
        'age': ('int', 'Age (years)')
    }
}
PRIMARY = {
    'heart': ['age', 'sex', 'cp', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca', 'exang'],
    'diabetes': ['age', 'glucose', 'bmi', 'blood_pressure', 'pregnancies']
}
REQUIRED = {'heart': ['age', 'sex'], 'diabetes': ['glucose'], 'cancer': []}
BIN = {'heart': ['sex', 'fbs', 'exang']}
PH = {'age': '55', 'trestbps': '140', 'chol': '240', 'thalach': '150', 'oldpeak': '1.5', 'glucose': '110', 'bmi': '28',
      'blood_pressure': '70', 'pregnancies': '2', 'insulin': '80', 'skin_thickness': '25', 'diabetes_pedigree': '0.5'}
STEP = {'oldpeak': 0.1, 'bmi': 0.1, 'diabetes_pedigree': 0.001}


@st.cache_resource
def load_all():
    return joblib.load(f'{PROJECT_DIR}/models/app_bundle.pkl'), joblib.load(f'{PROJECT_DIR}/data/splits.pkl')


@st.cache_data
def load_tables():
    return {k: pd.read_csv(f'{PROJECT_DIR}/outputs/{k}.csv') for k in ['cv_results', 'tuned_results', 'final_test_results', 'final_test_all_models']}


BUNDLE, SPLITS = load_all()
T = load_tables()


def spec(name, f):
    return SPECS.get(name, {}).get(f, ('num', f.capitalize()))


def short(label):
    return label.split(' (')[0]


def sig(values):
    return tuple((f, None if np.isnan(v) else v) for f, v in values.items())


def prepare(name, raw):
    v = dict(raw)
    for f in BIN.get(name, []):
        if np.isnan(v.get(f, np.nan)):
            v[f] = BUNDLE[name]['stats'][f]['median']
    return v


def predict(name, values):
    X = pd.DataFrame([prepare(name, values)], columns=BUNDLE[name]['features']).astype(float)
    return float(BUNDLE[name]['model'].predict_proba(X)[0, 1])


def effects(name, values, given):
    base = predict(name, values)
    return pd.Series({f: base - predict(name, {**values, f: np.nan}) for f in given})


def describe(name, f, v):
    kind, label, *rest = spec(name, f)
    if kind == 'cat':
        return f'{short(label)}: {rest[0].get(int(v), int(v))}'
    return f'{short(label)}: {v:g}'


def reset(name):
    for k in [k for k in st.session_state if k.startswith(f'{name}_')]:
        del st.session_state[k]


def load_patient(name):
    reset(name)
    X_te, y_te = SPLITS[name][1], SPLITS[name][3]
    i = np.random.randint(len(X_te))
    row, vals = X_te.iloc[i], {}
    for f in BUNDLE[name]['features']:
        v = row[f]
        vals[f] = np.nan if pd.isna(v) else float(v)
        if not pd.isna(v):
            st.session_state[f'{name}_{f}'] = int(v) if spec(name, f)[0] in ('cat', 'int') else float(v)
    given = [f for f, v in vals.items() if not np.isnan(v)]
    st.session_state[f'{name}_result'] = {'values': vals, 'given': given, 'truth': int(y_te.iloc[i])}


def field(name, f, col):
    kind, label, *rest = spec(name, f)
    s = BUNDLE[name]['stats'][f]
    key = f'{name}_{f}'
    if kind == 'cat':
        opts = rest[0]
        return col.selectbox(label, list(opts), index=None, format_func=opts.get, placeholder='Select', key=key)
    lo = 0.0 if s['min'] >= 0 else s['min'] * 1.5
    hi = s['max'] * 1.5
    ph = 'e.g. ' + PH.get(f, f"{s['median']:.4g}")
    if kind == 'int':
        return col.number_input(label, min_value=int(lo), max_value=int(np.ceil(hi)), value=None, step=1, placeholder=ph, key=key)
    step = STEP.get(f, 10.0 ** (int(np.floor(np.log10(max(hi - lo, 1e-9)))) - 3))
    dec = max(0, int(round(-np.log10(step))))
    return col.number_input(label, min_value=float(lo), max_value=float(hi), value=None, step=float(step), format=f'%.{dec}f', placeholder=ph, key=key)


def risk_bar(p, t):
    fig = go.Figure()
    fig.add_shape(type='rect', x0=0, x1=1, y0=0, y1=1, fillcolor=GREY, line_width=0)
    fig.add_shape(type='rect', x0=0, x1=p, y0=0, y1=1, fillcolor=RISK if p >= t else GREEN, line_width=0)
    fig.add_shape(type='line', x0=t, x1=t, y0=-0.25, y1=1.25, line=dict(color=INK, width=2))
    fig.add_annotation(x=t, y=1.55, text=f'flag threshold {t:.0%}', showarrow=False, font=dict(size=12, color=INK))
    fig.update_xaxes(range=[0, 1], tickformat='.0%', showgrid=False, zeroline=False)
    fig.update_yaxes(visible=False, range=[-0.4, 1.9])
    fig.update_layout(height=110, margin=dict(l=0, r=0, t=0, b=0), paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
    return fig


def tradeoff(name, t):
    y = SPLITS[name][3].values
    p = BUNDLE[name]['model'].predict_proba(SPLITS[name][1])[:, 1]
    grid = np.linspace(0.05, 0.95, 91)
    rec = [((p >= g) & (y == 1)).sum() / (y == 1).sum() for g in grid]
    fa = [((p >= g) & (y == 0)).sum() / (y == 0).sum() for g in grid]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=grid, y=rec, name='Sick patients caught', line=dict(color=GREEN, width=3)))
    fig.add_trace(go.Scatter(x=grid, y=fa, name='Healthy patients falsely flagged', line=dict(color=RISK, width=3)))
    fig.add_vline(x=t, line_color=INK, line_width=2)
    fig.update_yaxes(tickformat='.0%', range=[0, 1.02], gridcolor=GREY)
    fig.update_xaxes(title='Flag threshold', gridcolor=GREY)
    fig.update_layout(height=320, margin=dict(l=0, r=0, t=10, b=0), paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', legend=dict(orientation='h', y=-0.3))
    return fig, int(((p >= t) & (y == 1)).sum()), int((y == 1).sum()), int(((p >= t) & (y == 0)).sum()), int((y == 0).sum())


st.markdown("<div class='nav'><span class='brand'>Disease Risk Screening</span><span class='muted'>A machine-learning research project</span></div>", unsafe_allow_html=True)
name = st.radio('Condition', list(NAMES), format_func=NAMES.get, horizontal=True, label_visibility='collapsed')
feats = BUNDLE[name]['features']
tuned_t = BUNDLE[name]['threshold']
t = st.session_state.get(f'thr_{name}', round(tuned_t, 2))
st.markdown(f'## {NAMES[name]} prediction')
st.caption(POP[name])
tab1, tab2, tab3 = st.tabs(['Prediction', 'Model comparison', 'About'])

with tab1:
    left, right = st.columns([1.15, 1], gap='large')
    with left:
        with st.container(border=True, key='card_left'):
            st.markdown('#### Your measurements')
            req = ' and '.join(short(spec(name, f)[1]).lower() for f in REQUIRED[name])
            st.caption(f'Start with {req}. Add the details you have.' if req else 'Fill in what you have.')
            b1, b2 = st.columns(2)
            b1.button('Load sample patient', on_click=load_patient, args=(name,), key=f'sample_{name}')
            b2.button('Reset', on_click=reset, args=(name,), key=f'reset_{name}')
            primary = PRIMARY.get(name, [f for f in feats if f.startswith('mean')])
            more = [f for f in feats if f not in primary]
            raw = {}
            with st.form(f'form_{name}', border=False):
                cols = st.columns(2)
                for i, f in enumerate(primary):
                    raw[f] = field(name, f, cols[i % 2])
                with st.expander('More details (optional)'):
                    cols = st.columns(2)
                    for i, f in enumerate(more):
                        raw[f] = field(name, f, cols[i % 2])
                go_btn = st.form_submit_button('Get prediction')
            st.caption('Blank fields use typical values from the training data. Nothing you type is saved.')
            if go_btn:
                vals = {f: (np.nan if raw.get(f) is None else float(raw[f])) for f in feats}
                given = [f for f, v in vals.items() if not np.isnan(v)]
                miss = [short(spec(name, f)[1]) for f in REQUIRED[name] if f not in given]
                if miss:
                    st.error('Please enter: ' + ', '.join(miss))
                elif not given:
                    st.error('Enter at least one measurement.')
                else:
                    prev = st.session_state.get(f'{name}_result')
                    keep = prev is not None and sig(prev['values']) == sig(vals)
                    st.session_state[f'{name}_result'] = {'values': vals, 'given': given, 'truth': prev['truth'] if keep else None}

    with right:
        with st.container(border=True, key='card_right'):
            st.markdown('#### Prediction result')
            res = st.session_state.get(f'{name}_result')
            if res is None:
                st.markdown("<div class='empty'><div style='font-size:2rem'>&#8767;</div>Your prediction will appear here.<br>Enter measurements, then select Get prediction.</div>", unsafe_allow_html=True)
            else:
                values, given = res['values'], res['given']
                p = predict(name, values)
                flagged = p >= t
                c = RISK if flagged else GREEN
                head = f'Higher risk of {DISEASE[name]}' if flagged else f'Lower risk of {DISEASE[name]}'
                sub = 'The model flags this profile. A follow-up test or doctor visit is recommended.' if flagged else 'The model does not flag this profile. This does not rule anything out.'
                st.markdown(f"<div class='verdict' style='color:{c}'>{head}</div><div class='score'>Model score {p:.0%}</div>", unsafe_allow_html=True)
                st.write(sub)
                st.plotly_chart(risk_bar(p, t), width='stretch')
                st.caption(f'Based on {len(given)} of {len(feats)} inputs. The score is not a true probability.')
                if len(given) < 0.3 * len(feats):
                    st.warning('Few inputs given. The rest use typical values, so treat this score as weak.')
                out = [short(spec(name, f)[1]) for f in given if not BUNDLE[name]['stats'][f]['min'] <= values[f] <= BUNDLE[name]['stats'][f]['max']]
                if out:
                    st.warning('Outside the range seen in training: ' + ', '.join(out) + '. Interpret with caution.')
                if res['truth'] is not None:
                    st.info('Real outcome of this sample patient: ' + ('sick' if res['truth'] else 'not sick') + '.')
                with st.expander('Explain result'):
                    eff = effects(name, values, given)
                    top = eff.reindex(eff.abs().sort_values(ascending=False).head(6).index).iloc[::-1]
                    fig = go.Figure(go.Bar(x=top.values * 100, y=[describe(name, f, values[f]) for f in top.index], orientation='h',
                                           marker_color=[RISK if v > 0 else GREEN for v in top.values]))
                    fig.update_layout(height=280, margin=dict(l=0, r=0, t=10, b=0), paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
                                      xaxis_title='score points vs leaving this blank', showlegend=False)
                    fig.update_xaxes(zeroline=True, zerolinecolor=INK, gridcolor=GREY)
                    st.plotly_chart(fig, width='stretch')
                    st.caption('Red raises the score, green lowers it, compared with a typical patient.')
                    if name == 'cancer':
                        st.caption('Cancer measurements are highly correlated, so individual effects are unstable. Do not read them as causes.')
            with st.expander('Adjust sensitivity (advanced)'):
                st.slider('Flag threshold', 0.05, 0.95, round(tuned_t, 2), 0.01, key=f'thr_{name}')
                st.caption(f'Tuned value: {tuned_t:.2f}. Lower catches more sick patients but raises false alarms.')

with tab2:
    st.subheader('Sensitivity trade-off on held-out test patients')
    fig, caught, n_sick, fl, n_h = tradeoff(name, t)
    st.plotly_chart(fig, width='stretch')
    st.markdown(f'At threshold **{t:.2f}**: catches **{caught} of {n_sick}** sick patients and falsely flags **{fl} of {n_h}** healthy patients.')
    st.caption('The test set is small, so every number here has a margin of error of several points.')
    st.subheader('Algorithm comparison (5-fold cross-validation)')
    metric = st.selectbox('Compare by', ['roc_auc', 'recall', 'precision', 'f1', 'accuracy'], key=f'metric_{name}')
    cv = T['cv_results']
    cv = cv[cv.dataset == name]
    show = pd.DataFrame({'Model': [m + (' (selected)' if m == 'Logistic Regression' else '') for m in cv.model], 'Score': cv[metric].values})
    st.dataframe(show, hide_index=True, column_config={'Score': st.column_config.ProgressColumn('Score', min_value=0.0, max_value=1.0, format='%.3f')})
    st.caption('Differences smaller than about 0.05 are noise on datasets this small.')
    st.subheader('Effect of tuning (ROC-AUC)')
    tr = T['tuned_results']
    tr = tr[tr.dataset == name]
    st.dataframe(pd.DataFrame({'Model': tr.model.values, 'Before': tr.untuned_auc.values, 'After': tr.tuned_auc.values,
                               'Change': [f'{(b - a) * 100:+.1f} pp' for a, b in zip(tr.untuned_auc, tr.tuned_auc)]}), hide_index=True)
    st.subheader('Final test: all four tuned models at the default 0.5 cutoff')
    ta = T['final_test_all_models']
    st.dataframe(ta[ta.dataset == name].drop(columns='dataset'), hide_index=True)
    st.caption('Recall differences here mostly reflect how each model sets its probabilities, not better separation of patients.')
    st.subheader('Final test: selected model at both cutoffs')
    tf = T['final_test_results']
    st.dataframe(tf[tf.dataset == name].drop(columns='dataset'), hide_index=True)

with tab3:
    st.markdown("""
**Data.** UCI Heart Disease (Cleveland, 303 patients), Pima Indians Diabetes (768 women), Breast Cancer Wisconsin Diagnostic (569 samples).

**Method.** Missing values are imputed inside a pipeline fitted on training data only. Four algorithms (SVM, Logistic Regression, Random Forest, XGBoost) were compared with 5-fold cross-validation and tuned with grid search. Their scores were within noise, so the simplest and most explainable model, Logistic Regression, was chosen before the test set was opened. The flag threshold was tuned on out-of-fold predictions to reach a recall target.

**Limitations.**
Small datasets, so test results carry a wide margin of error.
Each model reflects its source population only.
Model scores are not calibrated probabilities.
Cancer: 2 of 42 malignant test samples were missed.
Educational project. Not a diagnostic tool, not clinically validated, and not medical advice.
""")

Overwriting /content/drive/MyDrive/ml_internship_project/repo/app.py


In [39]:
%%writefile /content/drive/MyDrive/ml_internship_project/repo/app.py
# Streamlit app: disease risk screening (v4, text inputs)
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import streamlit as st
import plotly.graph_objects as go

PROJECT_DIR = str(Path(__file__).parent)
GREEN, RISK, GREY, INK = '#1E5B45', '#B5472F', '#D5DDD7', '#1C2420'

st.set_page_config(page_title='Disease Risk Screening', layout='wide', initial_sidebar_state='collapsed')
st.markdown("""<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');
.stApp, .stApp p, .stApp label, .stApp input {font-family: 'Inter', sans-serif;}
[data-testid="stSidebar"], [data-testid="stSidebarCollapsedControl"], [data-testid="stHeader"], footer {display: none;}
.block-container {padding-top: 1.2rem; max-width: 1100px;}
.st-key-card_left, .st-key-card_right {background: #FFFFFF; border: 1px solid #C5D0C8 !important; border-radius: 16px; box-shadow: 0 2px 12px rgba(30,91,69,0.10); padding: 0.8rem 1rem;}
.stButton button {width: 100%; border-radius: 10px;}
[data-testid="stFormSubmitButton"] button {width: 100%; background: #1E5B45 !important; color: #FFFFFF !important; border: 0; border-radius: 10px; padding: 0.7rem 1rem;}
.nav {background: #1E5B45; border-radius: 14px; padding: 0.9rem 1.3rem; display: flex; justify-content: space-between; align-items: center; margin-bottom: 1.2rem;}
.brand {font-weight: 700; font-size: 1.2rem; color: #FFFFFF;}
.navtag {color: #CFE3D8; font-size: 0.85rem;}
.verdict {font-size: 1.8rem; font-weight: 700; line-height: 1.2;}
.score {font-size: 1.05rem; color: #3D4640; margin-top: 0.3rem;}
.empty {text-align: center; padding: 3rem 1rem; color: #6B746E;}
</style>""", unsafe_allow_html=True)

NAMES = {'heart': 'Heart disease', 'diabetes': 'Diabetes', 'cancer': 'Breast cancer'}
DISEASE = {'heart': 'heart disease', 'diabetes': 'diabetes', 'cancer': 'breast cancer'}
POP = {
    'heart': 'Trained on 303 patients from the Cleveland clinic (UCI), mostly older adults referred for suspected heart problems.',
    'diabetes': 'Trained on 768 women of Pima Indian heritage aged 21+. Not valid for men or other populations.',
    'cancer': 'Trained on 569 fine-needle-aspirate samples (Wisconsin). Inputs are lab measurements of cell nuclei.'
}
SPECS = {
    'heart': {
        'age': ('num', 'Age (years)'),
        'sex': ('cat', 'Sex', {0: 'Female', 1: 'Male'}),
        'cp': ('cat', 'Chest pain type', {1: 'Typical angina', 2: 'Atypical angina', 3: 'Non-anginal pain', 4: 'Asymptomatic'}),
        'trestbps': ('num', 'Resting blood pressure (mm Hg)'),
        'chol': ('num', 'Cholesterol (mg/dl)'),
        'fbs': ('cat', 'Fasting blood sugar above 120 mg/dl', {0: 'No', 1: 'Yes'}),
        'restecg': ('cat', 'Resting ECG', {0: 'Normal', 1: 'ST-T abnormality', 2: 'LV hypertrophy'}),
        'thalach': ('num', 'Maximum heart rate (bpm)'),
        'exang': ('cat', 'Exercise-induced angina', {0: 'No', 1: 'Yes'}),
        'oldpeak': ('num', 'ST depression (oldpeak)'),
        'slope': ('cat', 'ST slope', {1: 'Upsloping', 2: 'Flat', 3: 'Downsloping'}),
        'ca': ('cat', 'Major vessels (fluoroscopy)', {0: '0', 1: '1', 2: '2', 3: '3'}),
        'thal': ('cat', 'Thalassemia', {3: 'Normal', 6: 'Fixed defect', 7: 'Reversible defect'})
    },
    'diabetes': {
        'pregnancies': ('num', 'Pregnancies'),
        'glucose': ('num', 'Plasma glucose (mg/dl)'),
        'blood_pressure': ('num', 'Diastolic blood pressure (mm Hg)'),
        'skin_thickness': ('num', 'Skin fold thickness (mm)'),
        'insulin': ('num', 'Serum insulin (mu U/ml)'),
        'bmi': ('num', 'BMI'),
        'diabetes_pedigree': ('num', 'Diabetes pedigree function'),
        'age': ('num', 'Age (years)')
    }
}
PRIMARY = {
    'heart': ['age', 'sex', 'cp', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca', 'exang'],
    'diabetes': ['age', 'glucose', 'bmi', 'blood_pressure', 'pregnancies']
}
REQUIRED = {'heart': ['age', 'sex'], 'diabetes': ['glucose'], 'cancer': []}
BIN = {'heart': ['sex', 'fbs', 'exang']}
PH = {'age': '55', 'trestbps': '140', 'chol': '240', 'thalach': '150', 'oldpeak': '1.5', 'glucose': '110', 'bmi': '28',
      'blood_pressure': '70', 'pregnancies': '2', 'insulin': '80', 'skin_thickness': '25', 'diabetes_pedigree': '0.5'}


@st.cache_resource
def load_all():
    return joblib.load(f'{PROJECT_DIR}/models/app_bundle.pkl'), joblib.load(f'{PROJECT_DIR}/data/splits.pkl')


@st.cache_data
def load_tables():
    return {k: pd.read_csv(f'{PROJECT_DIR}/outputs/{k}.csv') for k in ['cv_results', 'tuned_results', 'final_test_results', 'final_test_all_models']}


BUNDLE, SPLITS = load_all()
T = load_tables()


def spec(name, f):
    return SPECS.get(name, {}).get(f, ('num', f.capitalize()))


def short(label):
    return label.split(' (')[0]


def wkey(name, f):
    return f'{name}_f4_{f}'


def sig(values):
    return tuple((f, None if np.isnan(v) else v) for f, v in values.items())


def prepare(name, raw):
    v = dict(raw)
    for f in BIN.get(name, []):
        if np.isnan(v.get(f, np.nan)):
            v[f] = BUNDLE[name]['stats'][f]['median']
    return v


def predict(name, values):
    X = pd.DataFrame([prepare(name, values)], columns=BUNDLE[name]['features']).astype(float)
    return float(BUNDLE[name]['model'].predict_proba(X)[0, 1])


def effects(name, values, given):
    base = predict(name, values)
    return pd.Series({f: base - predict(name, {**values, f: np.nan}) for f in given})


def describe(name, f, v):
    kind, label, *rest = spec(name, f)
    if kind == 'cat':
        return f'{short(label)}: {rest[0].get(int(v), int(v))}'
    return f'{short(label)}: {v:g}'


def reset(name):
    for k in [k for k in st.session_state if k.startswith(f'{name}_')]:
        del st.session_state[k]


def load_patient(name):
    reset(name)
    X_te, y_te = SPLITS[name][1], SPLITS[name][3]
    i = np.random.randint(len(X_te))
    row, vals = X_te.iloc[i], {}
    for f in BUNDLE[name]['features']:
        v = row[f]
        vals[f] = np.nan if pd.isna(v) else float(v)
        if not pd.isna(v):
            st.session_state[wkey(name, f)] = int(v) if spec(name, f)[0] == 'cat' else f'{v:g}'
    given = [f for f, v in vals.items() if not np.isnan(v)]
    st.session_state[f'{name}_result'] = {'values': vals, 'given': given, 'truth': int(y_te.iloc[i])}


def field(name, f, col):
    kind, label, *rest = spec(name, f)
    if kind == 'cat':
        opts = rest[0]
        return col.selectbox(label, list(opts), index=None, format_func=opts.get, placeholder='Select', key=wkey(name, f))
    ph = 'e.g. ' + PH.get(f, f"{BUNDLE[name]['stats'][f]['median']:.4g}")
    return col.text_input(label, placeholder=ph, key=wkey(name, f))


def risk_bar(p, t):
    fig = go.Figure()
    fig.add_shape(type='rect', x0=0, x1=1, y0=0, y1=1, fillcolor=GREY, line_width=0)
    fig.add_shape(type='rect', x0=0, x1=p, y0=0, y1=1, fillcolor=RISK if p >= t else GREEN, line_width=0)
    fig.add_shape(type='line', x0=t, x1=t, y0=-0.25, y1=1.25, line=dict(color=INK, width=2))
    fig.add_annotation(x=t, y=1.55, text=f'flag threshold {t:.0%}', showarrow=False, font=dict(size=12, color=INK))
    fig.update_xaxes(range=[0, 1], tickformat='.0%', showgrid=False, zeroline=False)
    fig.update_yaxes(visible=False, range=[-0.4, 1.9])
    fig.update_layout(height=110, margin=dict(l=0, r=0, t=0, b=0), paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
    return fig


def tradeoff(name, t):
    y = SPLITS[name][3].values
    p = BUNDLE[name]['model'].predict_proba(SPLITS[name][1])[:, 1]
    grid = np.linspace(0.05, 0.95, 91)
    rec = [((p >= g) & (y == 1)).sum() / (y == 1).sum() for g in grid]
    fa = [((p >= g) & (y == 0)).sum() / (y == 0).sum() for g in grid]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=grid, y=rec, name='Sick patients caught', line=dict(color=GREEN, width=3)))
    fig.add_trace(go.Scatter(x=grid, y=fa, name='Healthy patients falsely flagged', line=dict(color=RISK, width=3)))
    fig.add_vline(x=t, line_color=INK, line_width=2)
    fig.update_yaxes(tickformat='.0%', range=[0, 1.02], gridcolor=GREY)
    fig.update_xaxes(title='Flag threshold', gridcolor=GREY)
    fig.update_layout(height=320, margin=dict(l=0, r=0, t=10, b=0), paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', legend=dict(orientation='h', y=-0.3))
    return fig, int(((p >= t) & (y == 1)).sum()), int((y == 1).sum()), int(((p >= t) & (y == 0)).sum()), int((y == 0).sum())


st.markdown("<div class='nav'><span class='brand'>Disease Risk Screening</span><span class='navtag'>A machine-learning research project</span></div>", unsafe_allow_html=True)
name = st.radio('Condition', list(NAMES), format_func=NAMES.get, horizontal=True, label_visibility='collapsed')
feats = BUNDLE[name]['features']
tuned_t = BUNDLE[name]['threshold']
t = st.session_state.get(f'thr_{name}', round(tuned_t, 2))
st.markdown(f'## {NAMES[name]} prediction')
st.caption(POP[name])
tab1, tab2, tab3 = st.tabs(['Prediction', 'Model comparison', 'About'])

with tab1:
    left, right = st.columns([1.15, 1], gap='large')
    with left:
        with st.container(border=True, key='card_left'):
            st.markdown('#### Your measurements')
            req = ' and '.join(short(spec(name, f)[1]).lower() for f in REQUIRED[name])
            st.caption(f'Start with {req}. Add the details you have.' if req else 'Fill in what you have.')
            b1, b2 = st.columns(2)
            b1.button('Load sample patient', on_click=load_patient, args=(name,), key=f'sample_{name}')
            b2.button('Reset', on_click=reset, args=(name,), key=f'reset_{name}')
            primary = PRIMARY.get(name, [f for f in feats if f.startswith('mean')])
            more = [f for f in feats if f not in primary]
            raw = {}
            with st.form(f'form_{name}', border=False):
                cols = st.columns(2)
                for i, f in enumerate(primary):
                    raw[f] = field(name, f, cols[i % 2])
                if more:
                    with st.expander('More details (optional)'):
                        cols = st.columns(2)
                        for i, f in enumerate(more):
                            raw[f] = field(name, f, cols[i % 2])
                go_btn = st.form_submit_button('Get prediction', type='primary')
            st.caption('Type your values, then click Get prediction. Blank fields use typical values from the training data. Nothing is saved.')
            if go_btn:
                vals, bad = {}, []
                for f in feats:
                    v = raw.get(f)
                    if v is None or str(v).strip() == '':
                        vals[f] = np.nan
                        continue
                    try:
                        x = float(str(v).strip().replace(',', '.'))
                    except ValueError:
                        x = -1.0
                    if x < 0 or x > 3 * BUNDLE[name]['stats'][f]['max']:
                        bad.append(short(spec(name, f)[1]))
                        vals[f] = np.nan
                    else:
                        vals[f] = x
                given = [f for f, v in vals.items() if not np.isnan(v)]
                miss = [short(spec(name, f)[1]) for f in REQUIRED[name] if f not in given]
                if bad:
                    st.error('These values do not look right (numbers only, no negatives): ' + ', '.join(bad))
                elif miss:
                    st.error('Please enter: ' + ', '.join(miss))
                elif not given:
                    st.error('Enter at least one measurement.')
                else:
                    prev = st.session_state.get(f'{name}_result')
                    keep = prev is not None and sig(prev['values']) == sig(vals)
                    st.session_state[f'{name}_result'] = {'values': vals, 'given': given, 'truth': prev['truth'] if keep else None}

    with right:
        with st.container(border=True, key='card_right'):
            st.markdown('#### Prediction result')
            res = st.session_state.get(f'{name}_result')
            if res is None:
                st.markdown("<div class='empty'><div style='font-size:2rem'>&#8767;</div>Your prediction will appear here.<br>Enter measurements, then select Get prediction.</div>", unsafe_allow_html=True)
            else:
                values, given = res['values'], res['given']
                p = predict(name, values)
                flagged = p >= t
                c = RISK if flagged else GREEN
                head = f'Higher risk of {DISEASE[name]}' if flagged else f'Lower risk of {DISEASE[name]}'
                sub = 'The model flags this profile. A follow-up test or doctor visit is recommended.' if flagged else 'The model does not flag this profile. This does not rule anything out.'
                st.markdown(f"<div style='background:{c}14;border-left:6px solid {c};border-radius:10px;padding:1rem 1.2rem;margin-bottom:0.6rem'>"
                            f"<div class='verdict' style='color:{c}'>{head}</div><div class='score'>Model score {p:.0%}</div></div>", unsafe_allow_html=True)
                st.write(sub)
                st.plotly_chart(risk_bar(p, t), width='stretch')
                st.caption(f'Based on {len(given)} of {len(feats)} inputs. The score is not a true probability.')
                if len(given) < 0.3 * len(feats):
                    st.warning('Few inputs given. The rest use typical values, so treat this score as weak.')
                out = [short(spec(name, f)[1]) for f in given if not BUNDLE[name]['stats'][f]['min'] <= values[f] <= BUNDLE[name]['stats'][f]['max']]
                if out:
                    st.warning('Outside the range seen in training: ' + ', '.join(out) + '. Interpret with caution.')
                if res['truth'] is not None:
                    st.info('Real outcome of this sample patient: ' + ('sick' if res['truth'] else 'not sick') + '.')
                with st.expander('Explain result'):
                    eff = effects(name, values, given)
                    top = eff.reindex(eff.abs().sort_values(ascending=False).head(6).index).iloc[::-1]
                    fig = go.Figure(go.Bar(x=top.values * 100, y=[describe(name, f, values[f]) for f in top.index], orientation='h',
                                           marker_color=[RISK if v > 0 else GREEN for v in top.values]))
                    fig.update_layout(height=280, margin=dict(l=0, r=0, t=10, b=0), paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
                                      xaxis_title='score points vs leaving this blank', showlegend=False)
                    fig.update_xaxes(zeroline=True, zerolinecolor=INK, gridcolor=GREY)
                    st.plotly_chart(fig, width='stretch')
                    st.caption('Red raises the score, green lowers it, compared with a typical patient.')
                    if name == 'cancer':
                        st.caption('Cancer measurements are highly correlated, so individual effects are unstable. Do not read them as causes.')
            with st.expander('Adjust sensitivity (advanced)'):
                st.slider('Flag threshold', 0.05, 0.95, round(tuned_t, 2), 0.01, key=f'thr_{name}')
                st.caption(f'Tuned value: {tuned_t:.2f}. Lower catches more sick patients but raises false alarms.')

with tab2:
    st.subheader('Sensitivity trade-off on held-out test patients')
    fig, caught, n_sick, fl, n_h = tradeoff(name, t)
    st.plotly_chart(fig, width='stretch')
    st.markdown(f'At threshold **{t:.2f}**: catches **{caught} of {n_sick}** sick patients and falsely flags **{fl} of {n_h}** healthy patients.')
    st.caption('The test set is small, so every number here has a margin of error of several points.')
    st.subheader('Algorithm comparison (5-fold cross-validation)')
    metric = st.selectbox('Compare by', ['roc_auc', 'recall', 'precision', 'f1', 'accuracy'], key=f'metric_{name}')
    cv = T['cv_results']
    cv = cv[cv.dataset == name]
    show = pd.DataFrame({'Model': [m + (' (selected)' if m == 'Logistic Regression' else '') for m in cv.model], 'Score': cv[metric].values})
    st.dataframe(show, hide_index=True, column_config={'Score': st.column_config.ProgressColumn('Score', min_value=0.0, max_value=1.0, format='%.3f')})
    st.caption('Differences smaller than about 0.05 are noise on datasets this small.')
    st.subheader('Effect of tuning (ROC-AUC)')
    tr = T['tuned_results']
    tr = tr[tr.dataset == name]
    st.dataframe(pd.DataFrame({'Model': tr.model.values, 'Before': tr.untuned_auc.values, 'After': tr.tuned_auc.values,
                               'Change': [f'{(b - a) * 100:+.1f} pp' for a, b in zip(tr.untuned_auc, tr.tuned_auc)]}), hide_index=True)
    st.subheader('Final test: all four tuned models at the default 0.5 cutoff')
    ta = T['final_test_all_models']
    st.dataframe(ta[ta.dataset == name].drop(columns='dataset'), hide_index=True)
    st.caption('Recall differences here mostly reflect how each model sets its probabilities, not better separation of patients.')
    st.subheader('Final test: selected model at both cutoffs')
    tf = T['final_test_results']
    st.dataframe(tf[tf.dataset == name].drop(columns='dataset'), hide_index=True)

with tab3:
    st.markdown("""
**Data.** UCI Heart Disease (Cleveland, 303 patients), Pima Indians Diabetes (768 women), Breast Cancer Wisconsin Diagnostic (569 samples).

**Method.** Missing values are imputed inside a pipeline fitted on training data only. Four algorithms (SVM, Logistic Regression, Random Forest, XGBoost) were compared with 5-fold cross-validation and tuned with grid search. Their scores were within noise, so the simplest and most explainable model, Logistic Regression, was chosen before the test set was opened. The flag threshold was tuned on out-of-fold predictions to reach a recall target.

**Limitations.**
Small datasets, so test results carry a wide margin of error.
Each model reflects its source population only.
Model scores are not calibrated probabilities.
Cancer: 2 of 42 malignant test samples were missed.
Educational project. Not a diagnostic tool, not clinically validated, and not medical advice.
""")

Overwriting /content/drive/MyDrive/ml_internship_project/repo/app.py


In [40]:
# Check and download current app.py
from google.colab import files
print(open(f'{PROJECT_DIR}/repo/app.py').read().splitlines()[0])
files.download(f'{PROJECT_DIR}/repo/app.py')

# Streamlit app: disease risk screening (v4, text inputs)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>